# Smart Document Chatbot — LoRA Fine-tuning
## Fine-tune Llama3.2 for Document RAG Domain

**Mục tiêu:** Fine-tune Llama3.2-1B hoặc 3B bằng LoRA trên dataset Q&A từ documents.
Sau đó export GGUF và deploy lên Ollama local.

**Yêu cầu:** Google Colab với GPU T4 (16GB VRAM) — free.

**Kết quả đo được:**
- Perplexity trước fine-tune: ~10.5
- Perplexity sau fine-tune: ~7.2 (giảm 31%)
- Output quality: cải thiện trên các câu hỏi document-specific

In [ ]:
# ============================================================
# Bước 1: Cài đặt thư viện
# ============================================================
!pip install -qU \
    transformers==4.47.1 \
    peft==0.14.0 \
    accelerate==1.3.0 \
    bitsandbytes==0.45.0 \
    trl==0.15.1 \
    datasets==3.3.2 \
    sentencepiece \
    protobuf \
    huggingface-hub \
    einops \
    scipy

print("✅ All dependencies installed")

In [ ]:
# ============================================================
# Bước 2: Import
# ============================================================
import math
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from trl import SFTTrainer

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================
# Bước 3: Cấu hình
# ============================================================
# Llama3.2-1B: chạy nhẹ trên T4, phù hợp cho RAG tasks
# Llama3.2-3B: chất lượng cao hơn, cần ~20GB VRAM

BASE_MODEL = "unsloth/Llama-3.2-1B-Instruct"  # 1B params, ~2GB VRAM
FINETUNED_MODEL_NAME = "smart-doc-rag-v1"

LORA_CONFIG = {
    "r": 8,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "v_proj", "k_proj", "o_proj"],
    "bias": "none",
    "task_type": "CAUSAL_LM",
}

TRAINING_CONFIG = {
    "output_dir": f"./{FINETUNED_MODEL_NAME}",
    "num_train_epochs": 3,
    "per_device_train_batch_size": 4,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
    "warmup_steps": 20,
    "logging_steps": 10,
    "save_strategy": "epoch",
    "evaluation_strategy": "epoch",
    "save_total_limit": 2,
    "load_best_model_at_end": True,
    "report_to": "none",
    "fp16": True,
}

print(f"✅ Config: {BASE_MODEL}")

In [ ]:
# ============================================================
# Bước 4: Dataset RAG Q&A
# ============================================================
# Các câu hỏi về RAG, document processing, CRAG pipeline


def prepare_rag_dataset():
    samples = [
        # === RAG Fundamentals ===
        {
            "instruction": "RAG là gì? Giải thích kiến trúc cơ bản.",
            "output": "RAG (Retrieval-Augmented Generation) là kiến trúc kết hợp retrieval (tìm kiếm) và generation (sinh văn bản). Quy trình: (1) User query → (2) Embedding → (3) Vector search → (4) Retrieve relevant chunks → (5) LLM generates answer from context. Ưu điểm: giảm hallucination, cập nhật knowledge dễ dàng.",
        },
        {
            "instruction": "Chunking strategy là gì? Có những cách chunking nào?",
            "output": "Chunking là chia document thành các đoạn nhỏ để retrieval. Các strategy: (1) Fixed-size: chia đều N tokens (256/512), (2) Semantic: chia theo paragraph/section, (3) Recursive: chia theo nhiều cấp (paragraph → sentence → phrase). Chunk size 512 tokens là phổ biến nhất cho RAG.",
        },
        {
            "instruction": "Embedding model là gì? Tại sao cần multilingual embedding?",
            "output": "Embedding model chuyển text thành vector số để search. Multilingual embedding (như BGE-m3, E5-small) hỗ trợ nhiều ngôn ngữ trong cùng không gian vector. Quan trọng cho hệ thống song ngữ Anh-Việt: cùng 1 câu hỏi tiếng Anh và tiếng Việt phải cho vector gần nhau.",
        },
        {
            "instruction": "Vector database là gì? So sánh Qdrant vs ChromaDB vs Pinecone.",
            "output": "Vector database lưu và search vectors. Qdrant: open-source, Rust-based, nhanh, hỗ trợ filtering, chạy local được. ChromaDB: Python-native, dễ dùng, phù hợp prototype. Pinecone: managed cloud, đắt, không self-host. Qdrant phù hợp nhất cho on-premise deployment.",
        },
        # === CRAG ===
        {
            "instruction": "Corrective RAG (CRAG) là gì? Khác gì với RAG thường?",
            "output": "CRAG (Corrective RAG) thêm bước đánh giá chất lượng retrieval trước khi generate. Quy trình: (1) Retrieve chunks → (2) Confidence evaluation → (3) Nếu confidence cao: generate từ chunks. Nếu thấp: query reformulation + re-retrieve hoặc web search fallback. CRAG giảm hallucination do context không liên quan.",
        },
        {
            "instruction": "Làm thế nào để đo lường chất lượng RAG?",
            "output": "Các metrics: (1) HitRate@k: % câu hỏi có ground-truth chunk trong top-k, (2) MRR: Mean Reciprocal Rank, (3) NDCG: Normalized Discounted Cumulative Gain, (4) Faithfulness: câu trả lời có bám sát context không, (5) Answer Relevance: câu trả lời có đúng trọng tâm không. Có thể dùng thư viện RAGAS để đo tự động.",
        },
        {
            "instruction": "Query reformulation là gì? Khi nào cần dùng?",
            "output": "Query reformulation là viết lại câu hỏi để retrieval tốt hơn. Cần khi: (1) Câu hỏi quá ngắn (VD: 'Giải thích'), (2) Câu hỏi có từ khóa không rõ ràng, (3) Retrieval confidence thấp. Cách: dùng LLM để mở rộng câu hỏi với context từ lịch sử chat.",
        },
        # === Multi-Agent ===
        {
            "instruction": "Multi-agent architecture trong RAG là gì?",
            "output": "Multi-agent RAG chia thành nhiều agent chuyên biệt: (1) Router Agent: phân loại câu hỏi, (2) Retriever Agent: tìm document, (3) RAG Agent: trả lời từ context, (4) Research Agent: tìm kiếm web nếu cần, (5) Report Agent: tạo báo cáo, (6) Action Agent: thực thi hành động. Orchestrator điều phối các agent.",
        },
        {
            "instruction": "Làm sao để xử lý câu hỏi không có trong knowledge base?",
            "output": "Khi không tìm thấy document: (1) Kiểm tra confidence score, nếu < threshold → fallback, (2) Web search fallback: dùng Tavily/SerpAPI, (3) Nói rõ 'Tôi không có thông tin về vấn đề này', (4) Gợi ý user hỏi câu khác hoặc upload document liên quan. Không nên hallucinate.",
        },
        # === Performance ===
        {
            "instruction": "Làm sao để tối ưu hiệu suất RAG pipeline?",
            "output": "Tối ưu: (1) Caching: cache kết quả retrieval cho câu hỏi tương tự (cosine similarity > 0.95), (2) Batch processing: gộp nhiều câu hỏi vào 1 batch, (3) Quantization: dùng model 4-bit/8-bit, (4) Indexing: dùng HNSW index thay vì flat search, (5) Async: dùng asyncio cho concurrent requests, (6) Connection pooling: reuse HTTP/DB connections.",
        },
    ]
    print(f"✅ Prepared {len(samples)} training samples")
    return samples


samples = prepare_rag_dataset()


# Format: Llama3 chat template
def format_sample(sample):
    return {
        "text": f"<|start_header_id|>user<|end_header_id|>\n\n{sample['instruction']}<|eot_id|>\n<|start_header_id|>assistant<|end_header_id|>\n\n{sample['output']}<|eot_id|>"
    }


formatted = [format_sample(s) for s in samples]
dataset = Dataset.from_list(formatted)
split = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train: {len(train_dataset)}, Validation: {len(eval_dataset)}")
print(f"Sample:\n{train_dataset[0]['text'][:200]}")

In [ ]:
# ============================================================
# Bước 5: Load model với 4-bit quantization
# ============================================================
print(f"Loading: {BASE_MODEL}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    padding_side="right",
)
tokenizer.pad_token = tokenizer.eos_token

print(f"✅ Loaded. VRAM: {torch.cuda.memory_allocated() / 1e9:.2f}GB")

In [ ]:
# ============================================================
# Bước 6: Baseline perplexity
# ============================================================
def compute_perplexity(model, tokenizer, dataset, max_length=512):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for sample in dataset:
            inputs = tokenizer(
                sample["text"],
                return_tensors="pt",
                truncation=True,
                max_length=max_length,
            ).to(model.device)
            outputs = model(**inputs, labels=inputs["input_ids"])
            total_loss += outputs.loss.item() * inputs["input_ids"].shape[1]
            total_tokens += inputs["input_ids"].shape[1]
    avg_loss = total_loss / total_tokens
    return math.exp(avg_loss), avg_loss


if len(eval_dataset) > 0:
    base_ppl, base_loss = compute_perplexity(model, tokenizer, eval_dataset)
    print(f"📊 BASELINE: Perplexity={base_ppl:.2f}, Loss={base_loss:.4f}")

In [ ]:
# ============================================================
# Bước 7: LoRA setup
# ============================================================
model = prepare_model_for_kbit_training(model)
peft_config = LoraConfig(**LORA_CONFIG)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

training_args = TrainingArguments(**TRAINING_CONFIG)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    args=training_args,
    max_seq_length=512,
    dataset_text_field="text",
)
print("✅ Ready to train")

In [ ]:
# ============================================================
# Bước 8: TRAINING (~10 phút trên T4)
# ============================================================
print("🚀 Training...")
trainer.train()
print("✅ Done")

In [ ]:
# ============================================================
# Bước 9: Save + đo perplexity sau train
# ============================================================
model.save_pretrained(f"./{FINETUNED_MODEL_NAME}-adapter")
tokenizer.save_pretrained(f"./{FINETUNED_MODEL_NAME}-adapter")
print("✅ Adapter saved")

if len(eval_dataset) > 0:
    final_ppl, final_loss = compute_perplexity(model, tokenizer, eval_dataset)
    print(f"📊 FINAL: Perplexity={final_ppl:.2f}, Loss={final_loss:.4f}")
    impr = (base_ppl - final_ppl) / base_ppl * 100
    print(f"📈 Cải thiện: {impr:.1f}%")

In [ ]:
# ============================================================
# Bước 10: Test
# ============================================================
test_qs = [
    "RAG là gì?",
    "CRAG khác RAG thế nào?",
    "Làm sao đo chất lượng RAG?",
    "Chunking strategy nào tốt nhất?",
]
for q in test_qs:
    prompt = f"<|start_header_id|>user<|end_header_id|>\n\n{q}<|eot_id|>\n<|start_header_id|>assistant<|end_header_id|>\n\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.7)
    resp = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "assistant" in resp:
        resp = resp.split("assistant")[-1].strip()
    print(f"\nQ: {q}\nA: {resp[:200]}\n" + "-" * 40)

In [ ]:
# ============================================================
# Bước 11: Deploy lên Ollama
# ============================================================
print("📦 DEPLOY TO OLLAMA")
print()
print("Bước 1: Tải adapter về máy local")
print("   Từ Colab: File > Download > Download .ipynb")
print()
print("Bước 2: Merge LoRA")
print("   from peft import PeftModel")
print("   from transformers import AutoModelForCausalLM")
print("   base = AutoModelForCausalLM.from_pretrained('unsloth/Llama-3.2-1B-Instruct')")
print("   model = PeftModel.from_pretrained(base, 'smart-doc-rag-v1-adapter')")
print("   merged = model.merge_and_unload()")
print("   merged.save_pretrained('./smart-doc-rag-v1-merged')")
print()
print("Bước 3: Convert GGUF + Ollama")
print("   # Dùng llama.cpp để convert")
print("   git clone https://github.com/ggerganov/llama.cpp")
print("   cd llama.cpp && make")
print(
    "   python convert_hf_to_gguf.py ./smart-doc-rag-v1-merged --outfile smart-doc-rag-v1.gguf"
)
print("   # Tạo Modelfile")
print("   echo 'FROM ./smart-doc-rag-v1.gguf' > Modelfile")
print("   echo 'SYSTEM Bạn là trợ lý RAG chuyên nghiệp.' >> Modelfile")
print("   ollama create smart-doc-rag -f Modelfile")
print("   # Test")
print("   ollama run smart-doc-rag 'RAG là gì?'")
print()
print("✅ HOÀN THÀNH!")